<a href="https://colab.research.google.com/github/pamiemelo/atividade_extensionista_II/blob/main/Trabalho_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai-whisper

import whisper
import gradio as gr
from datetime import datetime
import os

print("Carregando o modelo Whisper (versão Tiny)...")
model = whisper.load_model("tiny")
print("Modelo carregado com sucesso!")

# Pasta para salvar os arquivos
os.makedirs("transcricoes_salvas", exist_ok=True)

# Função para listar os arquivos salvos que já estão na pasta
def obter_lista_arquivos():
    pasta = "transcricoes_salvas"
    if not os.path.exists(pasta):
        return []
    arquivos = [f for f in os.listdir(pasta) if f.endswith(".txt")]
    arquivos.sort(reverse=True)
    return arquivos

# Função para carregar o conteúdo do arquivo clicado na lista
def carregar_arquivo_por_nome(nome_arquivo):
    if not nome_arquivo:
        return "Selecione uma aula ao lado.", None

    caminho = os.path.join("transcricoes_salvas", nome_arquivo)
    if os.path.exists(caminho):
        with open(caminho, "r", encoding="utf-8") as f:
            conteudo = f.read()
        return conteudo, caminho
    return "Arquivo não encontrado.", None

# Função para atualizar a lista mantendo o que já foi salvo
def atualizar_componentes_biblioteca():
    arquivos = obter_lista_arquivos()
    return gr.update(choices=arquivos, value=arquivos[0] if arquivos else None)

# 2. Função de processamento de áudio
def processar_aula(audio_path, nome_arquivo, texto_atual, acao):
    if audio_path is None:
        return texto_atual, "⚠️ ATENÇÃO: Nenhum áudio foi gravado ou enviado.", gr.update(), None

    resultado = model.transcribe(
        audio_path,
        language="pt",
        temperature=0.0,
        beam_size=5
    )

    novo_texto_bruto = resultado["text"]
    frases = novo_texto_bruto.split(". ")
    novo_trecho = "\n\n".join([f"• {f.strip()}." for f in frases if f.strip()])

    if texto_atual and texto_atual.strip():
        texto_total = texto_atual + "\n\n" + novo_trecho
    else:
        texto_total = novo_trecho

    if acao == "Finalizar e Salvar na Biblioteca":
        if not nome_arquivo.strip():
            nome_arquivo = "aula_sem_nome"

        nome_limpo = "".join(c for c in nome_arquivo if c.isalnum() or c in (' ', '_', '-')).strip().replace(' ', '_')
        data_atual = datetime.now().strftime("%Y%m%d")
        caminho_arquivo = f"transcricoes_salvas/{nome_limpo}_{data_atual}.txt"

        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            f.write(f"=== TRANSCRIÇÃO DA AULA: {nome_arquivo} ===\n")
            f.write(f"Data: {datetime.now().strftime('%d/%m/%Y')}\n\n")
            f.write(texto_total)

        arquivos_atuais = obter_lista_arquivos()
        nome_arquivo_salvo = os.path.basename(caminho_arquivo)
        return texto_total, "✅ Salvo com sucesso na biblioteca!", gr.update(choices=arquivos_atuais, value=nome_arquivo_salvo), caminho_arquivo

    else:
        return texto_total, "✨ Transcrição adicionada! Pode enviar o próximo áudio.", gr.update(), None

# Função de limpeza
def limpar_tela_nova_aula(confirmacao):
    if not confirmacao:
        return "", "⚠️ Confirme a caixinha da Lixeira se realmente deseja limpar a tela."
    return "", "🗑️ Tela limpa e pronta para uma nova aula!"

# 3. Construção da Interface
tema = gr.themes.Monochrome(
    primary_hue="purple",
    secondary_hue="purple",
    neutral_hue="slate",
    font=["system-ui", "sans-serif"],
    font_mono=["Consolas", "monospace"]
)

with gr.Blocks() as demo:
    gr.Markdown("# Aula Transcrita+IA: Transcrição Inteligente para alunos oralizados ")

    # --- BLOCO 1: TRANSCRIÇÃO ---
    gr.Markdown("## 🎙️ Painel de Transcrição")
    with gr.Row():
        with gr.Column():
            input_nome = gr.Textbox(label="Nome da Aula", placeholder="Ex: Aula_Geografia")

            input_audio = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Áudio da Aula"
            )

            input_acao = gr.Radio(
                choices=["Continuar e inserir mais áudio", "Finalizar e Salvar na Biblioteca"],
                value="Continuar e inserir mais áudio",
                label="O que deseja fazer após transcrever este áudio?"
            )

            botao_executar = gr.Button("Processar Áudio", variant="primary")

        with gr.Column():
            saida_texto = gr.Textbox(label="Transcrição da Aula Atual", lines=12)
            saida_status = gr.Textbox(label="Status do Processo")

            gr.Markdown("---")
            checkbox_confirmar = gr.Checkbox(label="Tenho certeza que desejo limpar a tela atual", value=False)
            botao_limpar = gr.Button("🗑️ Limpar Tela")

    gr.Markdown("---")

    # --- BLOCO 2: BIBLIOTECA ---
    gr.Markdown("## 📚 Biblioteca de Aulas Salvas")
    gr.Markdown("Seus arquivos salvos anteriormente continuam aqui. Clique em qualquer aula da lista abaixo para visualizar e baixar:")

    with gr.Row():
        with gr.Column(scale=1):
            lista_aulas = gr.Radio(
                choices=obter_lista_arquivos(),
                label="Lista de Aulas Disponíveis",
                interactive=True
            )
            botao_atualizar = gr.Button("🔄 Atualizar Lista")

        with gr.Column(scale=2):
            conteudo_biblioteca = gr.Textbox(label="Transcrição da Aula Selecionada", lines=12, interactive=False)
            botao_baixar_selecionado = gr.DownloadButton("📥 Baixar esta Aula", variant="primary")

    # Conexões
    botao_executar.click(
        fn=processar_aula,
        inputs=[input_audio, input_nome, saida_texto, input_acao],
        outputs=[saida_texto, saida_status, lista_aulas, botao_baixar_selecionado]
    )

    botao_limpar.click(
        fn=limpar_tela_nova_aula,
        inputs=[checkbox_confirmar],
        outputs=[saida_texto, saida_status]
    )

    botao_atualizar.click(
        fn=atualizar_componentes_biblioteca,
        outputs=lista_aulas
    )

    lista_aulas.change(
        fn=carregar_arquivo_por_nome,
        inputs=lista_aulas,
        outputs=[conteudo_biblioteca, botao_baixar_selecionado]
    )

    # Carrega a lista automaticamente
    demo.load(fn=atualizar_componentes_biblioteca, outputs=lista_aulas)

# 4. Executa a interface
demo.launch(share=True, theme=tema)

Carregando o modelo Whisper (versão Tiny)...
Modelo carregado com sucesso!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e0cc57cba24173391.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
